# Step 9: is there a better free model than CLAP?

Same data, same splits, same small classifier (logistic regression); only the audio model that turns a clip into numbers changes.
Every model here is free to download and could run on the phone. Paid APIs and language models are out (Dr Yam, 18 Sep 2026).

| Model | What it was trained on | Size |
|---|---|---|
| CLAP htsat-unfused (our current choice) | sound and text pairs | 34 MB quantized on the phone |
| CLAP larger_clap_general | more sound and text pairs | larger |
| AST (MIT, AudioSet fine-tuned) | AudioSet, 527 sound classes including barks, growls, meows, purrs, hisses | about 87M weights |
| YAMNet (Google, the report's fallback) | AudioSet, 521 classes | 3.7M weights |
| PANNs CNN14 | AudioSet | about 80M weights |

Tasks: cats (CatMeows, 7-fold split by cat), dogs (EmotionalCanines official split, other dogs in test) for energy, mood and the report's four labels.
Scores are macro F1 (chance about 0.33 for three labels, 0.25 for four).

In [ ]:
import os, glob, json, time, zipfile, subprocess, urllib.request, shutil
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, librosa, torch
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.metrics import f1_score
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Data (same as steps 1 and 4)

In [ ]:
if not os.path.isdir("catmeows"):
    rec = json.load(urllib.request.urlopen("https://zenodo.org/api/records/4008297"))
    zf = [f for f in rec["files"] if f["key"].endswith(".zip")][0]
    urllib.request.urlretrieve(zf["links"]["self"], "catmeows.zip")
    zipfile.ZipFile("catmeows.zip").extractall("catmeows")
cat_files = sorted(glob.glob("catmeows/**/*.wav", recursive=True))
cat_y = np.array(["FBI".index(os.path.basename(f).split("_")[0]) for f in cat_files])
cat_g = np.array([os.path.basename(f).split("_")[1] for f in cat_files])

if not os.path.exists("EmotionalCanines/husky_train_labels.csv"):
    shutil.rmtree("EmotionalCanines", ignore_errors=True)
    subprocess.run("GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://github.com/tmdang1101/EmotionalCanines", shell=True, check=True)
rows = []
for breed in ["husky", "shiba"]:
    for split in ["train", "test"]:
        df = pd.read_csv(f"EmotionalCanines/{breed}_{split}_labels.csv"); df["split"] = split
        df["path"] = [f"EmotionalCanines/{breed}/{split}/{a}.wav" for a in df.audio_id]; rows.append(df)
dogs = pd.concat(rows, ignore_index=True)
MEDIA = "https://media.githubusercontent.com/media/tmdang1101/EmotionalCanines/main/"
def fetch(p):
    for _ in range(3):
        if os.path.exists(p) and os.path.getsize(p) > 1000: return
        try: urllib.request.urlretrieve(MEDIA + p[len("EmotionalCanines/"):], p)
        except Exception: time.sleep(2)
with ThreadPoolExecutor(8) as ex: list(ex.map(fetch, dogs.path))
def report_label(a, v):
    return 0 if v == "Positive" else 1 if v == "Neutral" else (3 if a == "High" else 2)
dog_y = {"energy": np.array([["Low","Medium","High"].index(a) for a in dogs.arousal]),
         "mood": np.array([["Negative","Neutral","Positive"].index(v) for v in dogs.valence]),
         "report4": np.array([report_label(a, v) for a, v in zip(dogs.arousal, dogs.valence)])}
tr = (dogs.split == "train").values
def load(paths, sr):  # first 10 s, like the phone
    return [librosa.load(p, sr=sr, mono=True)[0][:10 * sr] for p in paths]
print(len(cat_files), "cat clips,", len(dogs), "dog clips")

## 2. Scoring (identical for every model)

In [ ]:
GRID = {"logisticregression__C": [0.001, 0.01, 0.1, 1, 10], "logisticregression__class_weight": [None, "balanced"]}
def lr(): return make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
def score(Xc, Xd):
    r = {}
    # cats: C picked inside each training fold, 7 folds by cat
    pred = np.zeros_like(cat_y)
    for a, b in GroupKFold(7).split(Xc, cat_y, cat_g):
        g = GridSearchCV(lr(), GRID, scoring="f1_macro", cv=GroupKFold(5)).fit(Xc[a], cat_y[a], groups=cat_g[a])
        pred[b] = g.predict(Xc[b])
    r["cat"] = round(f1_score(cat_y, pred, average="macro"), 3)
    for t, y in dog_y.items():
        g = GridSearchCV(lr(), GRID, scoring="f1_macro", cv=StratifiedKFold(5, shuffle=True, random_state=42)).fit(Xd[tr], y[tr])
        r["dog_" + t] = round(f1_score(y[~tr], g.predict(Xd[~tr]), average="macro"), 3)
    return r
RESULTS = {}
def run(name, embed_fn, sr):
    t0 = time.time()
    Xc = np.stack([embed_fn(a) for a in load(cat_files, sr)])
    Xd = np.stack([embed_fn(a) for a in load(dogs.path, sr)])
    secs = (time.time() - t0) / (len(Xc) + len(Xd))
    r = score(Xc, Xd); r["dims"] = Xc.shape[1]; r["s_per_clip_colab_" + device] = round(secs, 3)
    RESULTS[name] = r
    print(name, r, flush=True)

## 3. The models

The two CLAP rows were run on 19 Nov 2026 (T4 GPU) before the Colab runtime timed out; their numbers are filled in here so the rest can run without repeating them. Set RERUN_CLAP = True to run them again.

In [ ]:
RERUN_CLAP = False
RESULTS["clap-htsat-unfused"] = {"cat": 0.527, "dog_energy": 0.536, "dog_mood": 0.482, "dog_report4": 0.384, "dims": 512}
RESULTS["larger_clap_general"] = {"cat": 0.533, "dog_energy": 0.547, "dog_mood": 0.45, "dog_report4": 0.387, "dims": 512}

In [ ]:
from transformers import ClapModel, ClapProcessor
for ckpt in ([] if not RERUN_CLAP else ["laion/clap-htsat-unfused", "laion/larger_clap_general"]):
    m = ClapModel.from_pretrained(ckpt).to(device).eval(); p = ClapProcessor.from_pretrained(ckpt)
    def emb(a, m=m, p=p):
        inp = p.feature_extractor([a], sampling_rate=48000, padding="repeatpad", return_tensors="pt")
        with torch.no_grad():
            e = m.audio_projection(m.audio_model(input_features=inp["input_features"].to(device)).pooler_output)
        return torch.nn.functional.normalize(e, dim=-1)[0].cpu().numpy()
    run(ckpt.split("/")[1], emb, 48000)
    del m; torch.cuda.empty_cache() if device == "cuda" else None

In [ ]:
from transformers import ASTFeatureExtractor, ASTModel
fe = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
ast = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593").to(device).eval()
def emb_ast(a):
    inp = fe(a, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        return ast(**{k: v.to(device) for k, v in inp.items()}).pooler_output[0].cpu().numpy()
run("ast-audioset", emb_ast, 16000)
del ast

In [ ]:
import tensorflow as tf, tensorflow_hub as hub
yam = hub.load("https://tfhub.dev/google/yamnet/1")
def emb_yam(a):
    _, e, _ = yam(a.astype(np.float32))
    e = e.numpy()
    return np.concatenate([e.mean(0), e.max(0)])
run("yamnet", emb_yam, 16000)

In [ ]:
subprocess.run("pip -q install panns-inference", shell=True)
from panns_inference import AudioTagging
pann = AudioTagging(checkpoint_path=None, device=device)
def emb_pann(a):
    _, e = pann.inference(a[None, :].astype(np.float32))
    return e[0]
run("panns-cnn14", emb_pann, 32000)

## 4. Summary

In [ ]:
df = pd.DataFrame(RESULTS).T
print(df.to_string())
print("RESULTS_JSON", json.dumps(RESULTS))